In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
textextract UniProt ID mappingtext(text)
from idmapping_selected.tab.gz inextracttextproteinIDtextGOannotation
text RefSeq ID find
"""

import gzip
import pandas as pd
import sys
from pathlib import Path
from collections import defaultdict
import time

# ====================================
# Path configuration
# ====================================
IDMAPPING_FILE = "/path/to/project/data_V3/Uniport/idmapping_selected.tab.gz"
PROTEIN_LIST_FILE = "/path/to/project/data_V3/Uniport/V7_catalog_prot4.txt"
OUTPUT_DIR = "/path/to/project/data_V3/Uniport/mapping_results"

# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Output files
MAPPING_OUTPUT = f"{OUTPUT_DIR}/protein_id_mapping.tsv"
GO_MAPPING_OUTPUT = f"{OUTPUT_DIR}/protein_GO_mapping.tsv"
STATS_OUTPUT = f"{OUTPUT_DIR}/mapping_statistics.txt"

# ====================================
# 1. readtextproteinIDlist
# ====================================
print("=" * 70)
print("text 1: readtextproteinIDlist")
print("=" * 70)

start_time = time.time()

try:
    # readtextcolumn, notext
    protein_df = pd.read_csv(PROTEIN_LIST_FILE, sep='\t', header=None, usecols=[0])
    user_proteins = set(protein_df[0].dropna().astype(str).unique())
    print(f"OK successfully readproteinIDfile")
    print(f" file: {PROTEIN_LIST_FILE}")
    print(f" total rows: {len(protein_df)}")
    print(f" uniqueproteinIDtext: {len(user_proteins)}")
    # textIDtext
    sample_ids = list(user_proteins)[:5]
    print(f" textID: {', '.join(sample_ids)}")
    # textIDtext
    if any(id.startswith('WP_') or id.startswith('YP_') or id.startswith('NP_') for id in sample_ids):
        print(f" OK texttoRefSeqformatID, textRefSeqcolumn(text4column)infind")
    elif any(id.startswith('Q') or id.startswith('P') or id.startswith('O') for id in sample_ids):
        print(f" OK texttoUniProtformatID, textUniProtKB-AC/IDcolumninfind")
        print(f" text: {time.time() - start_time:.2f} text\n")
        # savetextproteinlist
        user_protein_file = f"{OUTPUT_DIR}/user_protein_list.txt"
        with open(user_protein_file, 'w') as f:
            for prot in sorted(user_proteins):
                f.write(f"{prot}\n")
                print(f"OK textproteinlistsaved: {user_protein_file}\n")
except Exception as e:
    print(f"Failed error: notextreadproteinIDfile")
    print(f" {str(e)}")
    sys.exit(1)

# ====================================
# 2. process idmapping_selected.tab.gz
# ====================================
print("=" * 70)
print("text 2: extractmappingtext")
print("=" * 70)
print(f"processfile: {IDMAPPING_FILE}")
print("text, text...\n")

start_time = time.time()

# textmappingresults
mapping_data = []
go_data = []
match_count = 0
total_lines = 0

# column namestext(textREADME)
column_names = [
    'UniProtKB_AC', 'UniProtKB_ID', 'GeneID', 'RefSeq', 'GI', 'PDB', 'GO',
    'UniRef100', 'UniRef90', 'UniRef50', 'UniParc', 'PIR', 'NCBI_taxon',
    'MIM', 'UniGene', 'PubMed', 'EMBL', 'EMBL_CDS', 'Ensembl',
    'Ensembl_TRS', 'Ensembl_PRO', 'Additional_PubMed'
]

try:
    with gzip.open(IDMAPPING_FILE, 'rt', encoding='utf-8') as f:
        for line in f:
            total_lines += 1
            # textprocess100textrowstext
            if total_lines % 1000000 == 0:
                print(f" processed {total_lines:,} rows, found {match_count:,} matched...")
                parts = line.strip().split('\t')
                # textwithtextcolumn
                if len(parts) < 4:
                    continue
                # checktextIDcolumn
                uniprot_ac = parts[0] if len(parts) > 0 else ''
                uniprot_id = parts[1] if len(parts) > 1 else ''
                refseq = parts[3] if len(parts) > 3 else '' # RefSeqtext4column(text3)
                # textmatched
                is_match = False
                matched_id = None
                # check UniProtKB-AC and UniProtKB-ID
                if uniprot_ac in user_proteins:
                    is_match = True
                    matched_id = uniprot_ac
                elif uniprot_id in user_proteins:
                    is_match = True
                    matched_id = uniprot_id
                    # check RefSeq(textID, text)
                elif refseq:
                    # RefSeqcolumntextID, text "YP_031579.1" or "81941549; 49237298"
                    refseq_ids = [rid.strip() for rid in refseq.split(';')]
                    for rid in refseq_ids:
                        if rid in user_proteins:
                            is_match = True
                            matched_id = rid
                            break
                        if is_match:
                            match_count += 1
                            # textto22column
                            parts_extended = parts + [''] * (22 - len(parts))
                            # createmappingrecord
                            mapping_record = {col: parts_extended[i] for i, col in enumerate(column_names)}
                            mapping_record['Matched_ID'] = matched_id # textmatchedID
                            mapping_data.append(mapping_record)
                            # textwithGOannotation, textextract
                            go_terms = parts_extended[6] # text7columntextGO
                            if go_terms:
                                # GO terms text
                                for go_term in go_terms.split(';'):
                                    go_term = go_term.strip()
                                    if go_term:
                                        go_data.append({
                                            'Matched_ID': matched_id,
                                            'UniProtKB_AC': uniprot_ac,
                                            'UniProtKB_ID': uniprot_id,
                                            'RefSeq': refseq,
                                            'GO_Term': go_term
                                        })

                                        print(f"\nOK fileProcessing completed")
                                        print(f" total rows: {total_lines:,}")
                                        print(f" matchedrowstext: {match_count:,}")
                                        print(f" text: {time.time() - start_time:.2f} text\n")

except Exception as e:
    print(f"\nFailed error: processmappingfiletext")
    print(f" {str(e)}")
    sys.exit(1)

# ====================================
# 3. savemappingresults
# ====================================
print("=" * 70)
print("text 3: savemappingresults")
print("=" * 70)

if mapping_data:
    # savecomplete mappinginformation
    mapping_df = pd.DataFrame(mapping_data)
    mapping_df.to_csv(MAPPING_OUTPUT, sep='\t', index=False)
    print(f"OK complete mappingfilesaved: {MAPPING_OUTPUT}")
    print(f" recordtext: {len(mapping_df):,}\n")
    # textfirsttextrowstext
    print("mappingresultstext(first5rows): ")
    print(mapping_df[['Matched_ID', 'UniProtKB_AC', 'UniProtKB_ID', 'RefSeq', 'GO']].head())
    print()
    # saveGO mapping(textanalyze)
    if go_data:
        go_df = pd.DataFrame(go_data)
        go_df.to_csv(GO_MAPPING_OUTPUT, sep='\t', index=False)
        print(f"OK GO mappingfilesaved: {GO_MAPPING_OUTPUT}")
        print(f" recordtext: {len(go_df):,}")
        print(f" uniqueproteintext: {go_df['Matched_ID'].nunique():,}")
        print(f" unique GO terms: {go_df['GO_Term'].nunique():,}\n")
    else:
        print("Warning warning: not foundGOannotation\n")
else:
    print("Warning warning: not foundtextmatchedmappingtext\n")

# ====================================
# 4. generatestatistics report
# ====================================
print("=" * 70)
print("text 4: generatestatistics report")
print("=" * 70)

stats_lines = []
stats_lines.append("=" * 70)
stats_lines.append("UniProt ID mapping statistics report")
stats_lines.append("=" * 70)
stats_lines.append(f"Generated at: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")

stats_lines.append("Input data:")
stats_lines.append(f" user protein ID count: {len(user_proteins):,}")
stats_lines.append(f" total rows in mapping file: {total_lines:,}\n")

stats_lines.append("matchedresults:")
stats_lines.append(f" matchedmappingrecordtext: {match_count:,}")
if len(user_proteins) > 0:
    stats_lines.append(f" matchedtext: {match_count/len(user_proteins)*100:.2f}%\n")

if mapping_data:
    mapping_df = pd.DataFrame(mapping_data)
    stats_lines.append("Mapping details:")
    stats_lines.append(f" uniquematchedproteinID: {mapping_df['Matched_ID'].nunique():,}")
    stats_lines.append(f" with GeneID: {mapping_df['GeneID'].notna().sum():,}")
    stats_lines.append(f" with RefSeq: {mapping_df['RefSeq'].notna().sum():,}")
    stats_lines.append(f" with GO annotation: {mapping_df['GO'].notna().sum():,}")
    stats_lines.append(f" with Ensembl: {mapping_df['Ensembl'].notna().sum():,}\n")

if go_data:
    go_df = pd.DataFrame(go_data)
    stats_lines.append("GO annotation statistics:")
    stats_lines.append(f" total GO mapping records: {len(go_df):,}")
    stats_lines.append(f" with GO annotationprotein: {go_df['Matched_ID'].nunique():,}")
    stats_lines.append(f" unique GO terms: {go_df['GO_Term'].nunique():,}")
    # calculatecoveragetext
    proteins_with_go = go_df['Matched_ID'].nunique()
    coverage = proteins_with_go / len(user_proteins) * 100
    stats_lines.append(f" GO annotation coverage: {coverage:.2f}%\n")
    # GOtext
    stats_lines.append("GO term examples (first 10):")
    for go_term in go_df['GO_Term'].unique()[:10]:
        stats_lines.append(f" - {go_term}")

stats_lines.append("\n" + "=" * 70)
stats_lines.append("Output files:")
stats_lines.append(f" complete mapping: {MAPPING_OUTPUT}")
stats_lines.append(f" GO mapping: {GO_MAPPING_OUTPUT}")
stats_lines.append(f" statistics report: {STATS_OUTPUT}")
stats_lines.append("=" * 70)

# savestatistics report
stats_text = '\n'.join(stats_lines)
with open(STATS_OUTPUT, 'w', encoding='utf-8') as f:
    f.write(stats_text)

print(stats_text)
print(f"\nOK statistics reportsaved: {STATS_OUTPUT}")

# ====================================
# 5. textmatchedproteinanalyze
# ====================================
if mapping_data:
    mapped_proteins = set(mapping_df['Matched_ID'].unique())
    unmapped_proteins = user_proteins - mapped_proteins
    if unmapped_proteins:
        unmapped_file = f"{OUTPUT_DIR}/unmapped_proteins.txt"
        with open(unmapped_file, 'w') as f:
            for prot in sorted(unmapped_proteins):
                f.write(f"{prot}\n")
                print(f"\nWarning with {len(unmapped_proteins):,} proteinsnot foundmapping")
                print(f" textmappinglistsaved: {unmapped_file}")
                print(f" textmappingtext: {len(unmapped_proteins)/len(user_proteins)*100:.2f}%")
                if len(unmapped_proteins) <= 10:
                    print(" textmappingproteinIDtext:")
                    for prot in list(sorted(unmapped_proteins))[:10]:
                        print(f" - {prot}")

print("\n" + "=" * 70)
print("mappingextractcompleted!")
print("=" * 70)
print("\ntext: text GO textanalyze")
print(f"GO mappingfilepath: {GO_MAPPING_OUTPUT}\n")